## Load libraries

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import STL
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import ParameterGrid
from tqdm import tqdm

## Config

In [ ]:
TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "LocalAuthority"

TRAIN_START_DATE = pd.Timestamp("2007-04-01")
TRAIN_END_DATE   = pd.Timestamp("2022-03-31")

# rolling CV
ROLLING_TRAIN_WINDOW = 120   # 10 years
ROLLING_VAL_WINDOW   = 12    # 1 year

# lag plan
lag_combinations = [
    [1, 12],
    [1, 2, 12],
    [1, 2, 3, 12],
    [1, 2, 3, 4, 5, 6, 12],
    [1, 12, 24],
    [1, 2, 12, 24],
    [1, 2, 3, 12, 24],
    [1, 2, 3, 4, 5, 6, 12, 24],
]
all_lags = sorted({l for combo in lag_combinations for l in combo})

# RF tuning grid
rf_param_grid = {
    "n_estimators": [250, 500],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 0.5],
    "bootstrap": [True],
}

## Evaluation metric functions

In [ ]:
def mae(y, yhat): return np.mean(np.abs(y - yhat))
def rmse(y, yhat): return np.sqrt(np.mean((y-yhat)**2))

def smape(y, yhat, eps=1e-8):
    return 100*np.mean(2*np.abs(yhat-y)/(np.abs(y)+np.abs(yhat)+eps))

def mase(y, yhat, y_train, m=12, eps=1e-8):
    return np.mean(np.abs(y-yhat)) / (np.mean(np.abs(y_train[m:]-y_train[:-m]))+eps)


## Load data

In [ ]:
data = pd.read_excel("../../data/full_data.xlsx")
df = pd.read_excel("../../data/full_data.xlsx", parse_dates=[TIME_COL])
df = df.sort_values([ENTITY_COL, TIME_COL]).reset_index(drop=True)

# restrict to CV window only
mask = (df[TIME_COL] >= TRAIN_START_DATE) & (df[TIME_COL] <= TRAIN_END_DATE)
df_tv = df.loc[mask].copy().reset_index(drop=True)

dates = sorted(df_tv[TIME_COL].unique())

## Training with STL + Rolling CV

In [ ]:
results = []

for lag_set in lag_combinations:
    print(f"\nLag set: {lag_set}")
    
    for params in ParameterGrid(rf_param_grid):
        fold_MAE, fold_RMSE, fold_sMAPE, fold_MASE = [], [], [], []

        start_idx = ROLLING_TRAIN_WINDOW
        while True:
            train_end_idx = start_idx
            val_start_idx = train_end_idx
            val_end_idx   = val_start_idx + ROLLING_VAL_WINDOW

            if val_end_idx > len(dates):
                break

            train_start = dates[train_end_idx - ROLLING_TRAIN_WINDOW]
            train_end   = dates[train_end_idx - 1]
            val_start   = dates[val_start_idx]
            val_end     = dates[val_end_idx   - 1]

            mask_train = (df_tv[TIME_COL] >= train_start) & (df_tv[TIME_COL] <= train_end)
            mask_val   = (df_tv[TIME_COL] >= val_start)   & (df_tv[TIME_COL] <= val_end)

            fold_train = df_tv.loc[mask_train].copy()
            fold_val   = df_tv.loc[mask_val  ].copy()

            # =====================================
            # LEAK-FREE STL (PER FOLD)
            # =====================================
            fold_train["stl_trend"] = np.nan
            fold_train["stl_seasonal"] = np.nan
            fold_train["stl_resid"] = np.nan

            for la, sub in fold_train.groupby(ENTITY_COL):
                stl = STL(sub[TARGET_COL], period=12, robust=True)
                res = stl.fit()
                fold_train.loc[sub.index, "stl_trend"]    = res.trend
                fold_train.loc[sub.index, "stl_seasonal"] = res.seasonal
                fold_train.loc[sub.index, "stl_resid"]    = res.resid

            # extend STL into validation
            fold_val["stl_resid"] = 0.0
            for la, sub in fold_train.groupby(ENTITY_COL):
                last_season = sub["stl_seasonal"].values[-12:]
                last_trend  = sub["stl_trend"].values
                t_idx = np.arange(len(last_trend))
                coef = np.polyfit(t_idx, last_trend, 1)

                future_idx = fold_val[fold_val[ENTITY_COL]==la].index
                k = np.arange(len(future_idx)) + len(t_idx)

                fold_val.loc[future_idx, "stl_trend"] = coef[0]*k + coef[1]
                fold_val.loc[future_idx, "stl_seasonal"] = last_season[:len(future_idx)]

            # =====================================
            # CREATE LAGS
            # =====================================
            for lag in lag_set:
                for comp in ["stl_trend","stl_seasonal","stl_resid"]:
                    col = f"{comp}_lag{lag}"
                    fold_train[col] = fold_train.groupby(ENTITY_COL)[comp].shift(lag)
                    fold_val[col]   = fold_val.groupby(ENTITY_COL)[comp].shift(lag)

            lag_cols = [
                f"{c}_lag{l}" for c in ["stl_trend","stl_seasonal","stl_resid"] for l in lag_set
            ]

            fold_train = fold_train.dropna(subset=lag_cols)
            fold_val   = fold_val.dropna(subset=lag_cols)

            if fold_train.empty or fold_val.empty:
                start_idx += ROLLING_VAL_WINDOW
                continue

            scale_cols = continuous_cols + lag_cols
            scaler = StandardScaler()
            fold_train[scale_cols] = scaler.fit_transform(fold_train[scale_cols])
            fold_val[scale_cols]   = scaler.transform(fold_val[scale_cols])

            X_train = fold_train[continuous_cols + categorical_cols + lag_cols]
            y_train = fold_train[TARGET_COL].values
            X_val   = fold_val  [continuous_cols + categorical_cols + lag_cols]
            y_val   = fold_val  [TARGET_COL].values

            rf = RandomForestRegressor(**params, n_jobs=-1, random_state=42)
            rf.fit(X_train, y_train)
            y_pred = rf.predict(X_val)

            fold_MAE.append(mae(y_val,y_pred))
            fold_RMSE.append(rmse(y_val,y_pred))
            fold_sMAPE.append(smape(y_val,y_pred))
            fold_MASE.append(mase(y_val,y_pred,y_train))

            start_idx += ROLLING_VAL_WINDOW

        if fold_MAE:
            results.append({
                "model":"RandomForest",
                "lag_set":tuple(lag_set),
                "params":params,
                "MAE":np.mean(fold_MAE),
                "RMSE":np.mean(fold_RMSE),
                "sMAPE":np.mean(fold_sMAPE),
                "MASE":np.mean(fold_MASE),
                "folds":len(fold_MAE)
            }) 

## Results

In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSE")
results_df.to_csv("rf_leakfree_cv_results.csv", index=False)
print(results_df.head(10))